#### farm_master 생성 노트북 (Blob Storage)
##### 축종 코드로 축종 전처리

현재 테스트 때문에 조사날짜 고정중 
2번 파트에서 수정가능

In [ ]:
import sys
sys.path.append("/Workspace/방역로/00_Shared_Utils")
from utils_config import CATALOG,BASE_PATH

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType
import math

In [ ]:
# ──────────────────────────────────────────
# 1. 축종코드 로드 → {세부구분: 상위축종} 딕셔너리
# ──────────────────────────────────────────
code_dict = {
    r["detail"]: r["parent_name"]
    for r in (
        spark.read
        .option("header", True)
        .csv("/Volumes/dt4_team1_databricks/raw/test/livestock_codes.csv")
        .filter(F.col("상세구분").isNotNull() & (F.col("상세구분") != ""))
        .select(F.col("상세구분").alias("detail"), F.col("축종명").alias("parent_name"))
        .collect()
    )
}

In [ ]:
@F.udf()
def normalize_livestock(name):
    if not name:
        return name
    parts = [p.strip() for p in name.split("/")]
    parents = {code_dict.get(p, p) for p in parts}
    return parents.pop() if len(parents) == 1 else name

@F.udf()
def normalize_type(name, ltype):
    if not name:
        return ltype
    parts = [p.strip() for p in name.split("/")]
    return name if all(p in code_dict for p in parts) else ltype

In [ ]:
# ── 2. farm_status 전처리 후 저장 ──
(
    spark.read.table("dt4_team1_databricks.raw.farm_status")
    .select(
        F.split(F.coalesce("소재지지번주소", "소재지도로명주소"), " ")[0].alias("city"),
        F.col("시군명").alias("county"),
        F.col("농장명").alias("farm_name"),
        F.coalesce("소재지지번주소", "소재지도로명주소").alias("farm_address"),
        F.col("WGS84위도").cast("double").alias("latitude"),
        F.col("WGS84경도").cast("double").alias("longitude"),
        F.col("축종명").alias("_name"),
        F.col("상세구분").alias("_type"),
        F.col("사육두수(마리)").cast("int").alias("head_count"),
        F.lit("2026-06-29").cast("date").alias("reference_date"),
        # F.current_date().alias("survey_date"),
    )
    .withColumn("livestock_name", normalize_livestock("_name"))
    .withColumn("livestock_type", normalize_type("_name", "_type"))
    .drop("_name", "_type")
    .withColumn("outbreak_date", F.lit(None).cast("date"))
    .withColumn("label_infected", F.lit(0))
    .withColumn("farm_id", F.md5(F.concat_ws("|", "farm_name", "farm_address")))
    .withColumn("INGESTED_AT", F.current_timestamp())
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.farm_master")
)